In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
REY_DIR = DATA_DIR / "REY_DATASET"
REY_SCAN_DIR = REY_DIR / "REY_scan_anonim"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CSV_LIMPIO = TABLES_DIR / "rey_traza_limpia.csv"

print("TensorFlow:", tf.__version__)
print("CSV limpio existe:", CSV_LIMPIO.exists())
print("REY_SCAN_DIR existe:", REY_SCAN_DIR.exists())

In [ ]:
df = pd.read_csv(CSV_LIMPIO)

print("Número de muestras:", len(df))
display(df.head())

print("\nDistribución de rotaciones:")
print(df["rot"].value_counts().sort_index())

In [ ]:
IMG_SIZE = 224

RANDOM_STATE = 7

print("Tamaño de entrada del modelo:", IMG_SIZE, "x", IMG_SIZE)

In [ ]:
def cargar_imagen_y_bbox(fila):
    """
    Lee una imagen original y devuelve:
    - imagen redimensionada y normalizada
    - bounding box normalizada [x, y, w, h]
    """
    
    img_path = REY_SCAN_DIR / fila["forig"]
    
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        raise ValueError("No se pudo leer la imagen: " + str(img_path))
    
    alto_original, ancho_original = img.shape
    
    # Bounding box en píxeles originales
    x = float(fila["x"])
    y = float(fila["y"])
    h = float(fila["h"])
    w = float(fila["w"])
    
    # Normalizar la bounding box respecto al tamaño original
    x_norm = x / ancho_original
    y_norm = y / alto_original
    w_norm = w / ancho_original
    h_norm = h / alto_original
    
    bbox = np.array([x_norm, y_norm, w_norm, h_norm], dtype=np.float32)
    
    # Redimensionar imagen para la CNN
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # Normalizar imagen a [0,1]
    img_resized = img_resized.astype(np.float32) / 255.0
    
    # Añadir canal: (224,224) -> (224,224,1)
    img_resized = np.expand_dims(img_resized, axis=-1)
    
    return img_resized, bbox

In [ ]:
X = []
y = []
nombres = []

for i, fila in df.iterrows():
    img, bbox = cargar_imagen_y_bbox(fila)
    
    X.append(img)
    y.append(bbox)
    nombres.append(fila["forig"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
nombres = np.array(nombres)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Ejemplo y[0]:", y[0])

In [ ]:
X_train, X_temp, y_train, y_temp, names_train, names_temp = train_test_split(
    X, y, nombres,
    test_size=0.30,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test, names_val, names_test = train_test_split(
    X_temp, y_temp, names_temp,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    
    # 4 salidas: x, y, w, h
    # sigmoid porque queremos valores entre 0 y 1
    layers.Dense(4, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
checkpoint_path = MODELS_DIR / "rey_bbox_keras_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss MSE")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["mae"], label="train")
plt.plot(history.history["val_mae"], label="val")
plt.title("MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.tight_layout()

fig_path = FIGURES_DIR / "rey_bbox_training_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
def calcular_iou(box1, box2):
    """
    box = [x, y, w, h] normalizado
    """
    
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    # Pasar a formato x1,y1,x2,y2
    box1_x1 = x1
    box1_y1 = y1
    box1_x2 = x1 + w1
    box1_y2 = y1 + h1
    
    box2_x1 = x2
    box2_y1 = y2
    box2_x2 = x2 + w2
    box2_y2 = y2 + h2
    
    inter_x1 = max(box1_x1, box2_x1)
    inter_y1 = max(box1_y1, box2_y1)
    inter_x2 = min(box1_x2, box2_x2)
    inter_y2 = min(box1_y2, box2_y2)
    
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    
    area_inter = inter_w * inter_h
    
    area1 = w1 * h1
    area2 = w2 * h2
    
    area_union = area1 + area2 - area_inter
    
    if area_union == 0:
        return 0
    
    return area_inter / area_union

In [ ]:
y_pred = model.predict(X_test)

# Aseguramos que no se salgan de [0,1]
y_pred = np.clip(y_pred, 0, 1)

ious = []
maes = []

for i in range(len(y_test)):
    iou = calcular_iou(y_test[i], y_pred[i])
    ious.append(iou)
    
    mae = np.mean(np.abs(y_test[i] - y_pred[i]))
    maes.append(mae)

ious = np.array(ious)
maes = np.array(maes)

print("IoU media:", ious.mean())
print("IoU std:", ious.std())
print("MAE medio bbox normalizada:", maes.mean())
print("MAE std:", maes.std())

print("Porcentaje IoU > 0.5:", np.mean(ious > 0.5))
print("Porcentaje IoU > 0.7:", np.mean(ious > 0.7))

In [ ]:
resultados_bbox = pd.DataFrame({
    "imagen": names_test,
    "x_real": y_test[:, 0],
    "y_real": y_test[:, 1],
    "w_real": y_test[:, 2],
    "h_real": y_test[:, 3],
    "x_pred": y_pred[:, 0],
    "y_pred": y_pred[:, 1],
    "w_pred": y_pred[:, 2],
    "h_pred": y_pred[:, 3],
    "iou": ious,
    "mae": maes
})

csv_resultados = TABLES_DIR / "rey_bbox_keras_resultados_test.csv"
resultados_bbox.to_csv(csv_resultados, index=False)

print("Resultados guardados en:", csv_resultados)
display(resultados_bbox.head())

In [ ]:
def dibujar_bbox_normalizada(img_color, bbox, color, grosor=2):
    """
    Dibuja bbox normalizada [x,y,w,h] sobre una imagen ya cargada.
    """
    alto, ancho = img_color.shape[:2]
    
    x = int(bbox[0] * ancho)
    y = int(bbox[1] * alto)
    w = int(bbox[2] * ancho)
    h = int(bbox[3] * alto)
    
    cv2.rectangle(img_color, (x, y), (x + w, y + h), color, grosor)
    
    return img_color

In [ ]:
n_ver = 6
indices = np.arange(len(X_test))

plt.figure(figsize=(12, 8))

for i in range(n_ver):
    idx = indices[i]
    
    img_path = REY_SCAN_DIR / names_test[idx]
    img_color = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    
    img_color = dibujar_bbox_normalizada(img_color, y_test[idx], (0, 0, 255), 3)   # rojo real
    img_color = dibujar_bbox_normalizada(img_color, y_pred[idx], (0, 255, 0), 3)   # verde predicho
    
    img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_rgb)
    plt.title(f"IoU={ious[idx]:.2f}")
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "rey_bbox_predicciones_keras.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)